# TestClient (sync)

This notebook covers:

1. What `fastapi.testclient.TestClient` actually is (httpx pointed at the ASGI app in-process)
2. The three layers every API test asserts on: status, headers, body
3. Pytest fixtures to centralize setup, with `conftest.py` for cross-file reuse
4. `@pytest.mark.parametrize` for tabular happy/sad-path coverage
5. Markers and selection — separating a fast inner-loop suite from a slower CI tier

**Scope**: FastAPI + Pydantic v2 + pytest, executed with `TestClient` (no Uvicorn, no network). All test files are written to a fresh temp directory and run via `python -m pytest` as a subprocess, so you see the actual pytest output a CI run would produce.

## 1. What `TestClient` Actually Is

`TestClient` is an HTTP client that talks to your FastAPI app **in-process**. There is no Uvicorn, no socket, no port, no localhost. Under the hood it is an `httpx.Client` configured with an ASGI transport that hands requests directly to your ASGI app object. That has two consequences worth understanding before you write a single test:

- **It's fast.** No TCP handshake, no JSON over the wire, no process-boundary serialization. A test that exercises 30 endpoints completes in tens of milliseconds. That speed is what makes a tight TDD loop feel cheap enough to use.
- **It uses a synchronous API even when your endpoints are `async def`.** `TestClient` runs the event loop for you on each call. You write `response = client.get("/assets")` regardless of whether `get_assets` is `def` or `async def`. The async client in notebook 7.2 is only worth reaching for when the *test itself* needs to fire concurrent requests.

The trick to think clearly about TestClient: it's just `httpx`. Anything you can do with `httpx.Client` — auth, headers, cookies, follow_redirects, timeouts — you can do here, because it *is* an `httpx.Client`.

In [1]:
import httpx
from fastapi import FastAPI
from fastapi.testclient import TestClient

tiny_app = FastAPI()

@tiny_app.get("/")
def root():
    return {"status": "ok"}

client = TestClient(tiny_app)
response = client.get("/")

print("type        :", type(response).__name__, "(from", type(response).__module__ + ")")
print("status_code :", response.status_code)
print("headers     :", dict(response.headers))
print("body (json) :", response.json())
print("url         :", response.url)
print("is httpx?   :", isinstance(response, httpx.Response))

type        : Response (from httpx)
status_code : 200
headers     : {'content-length': '15', 'content-type': 'application/json'}
body (json) : {'status': 'ok'}
url         : http://testserver/
is httpx?   : True


C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


The response object is `httpx.Response` — the same class your production code would see if it called your service for real. That's the point: tests speak the language a client speaks.

A subtle behavior to internalize early: by default, `TestClient` **re-raises server-side exceptions in the test process** instead of returning a 500. That's usually what you want — a stack trace beats a 500 body when you're debugging. Pass `raise_server_exceptions=False` when you specifically want to assert on the *response* a real client would receive (see notebook 6.1 section 6).

## 2. Anatomy of a Pytest Test

A pytest test is, structurally, just a function:

- It lives in a file whose name starts with `test_` (or ends with `_test.py`).
- Its own name starts with `test_`.
- It contains one or more `assert` statements.

No base class, no special decorator, no return value. Pytest discovers files matching `test_*.py`, imports them, finds the functions, and runs each as an isolated test. Failures are reported by reading the locals at the failing `assert` — that's where pytest's famously good error messages come from.

Below we write a test-shaped function and call it directly inside the notebook, just to see the shape. Section 4 graduates to running real pytest against a file.

In [2]:
def test_root_returns_ok():
    # GIVEN a client wired to our app
    response = client.get("/")
    # WHEN/THEN
    assert response.status_code == 200
    assert response.json() == {"status": "ok"}

test_root_returns_ok()
print("PASSED")

PASSED


The Given/When/Then comments are optional — many teams skip them — but the underlying *shape* is universal: set up the world, perform one action, assert on the observable outcome. Tests that try to assert on too many actions at once become difficult to debug; one logical assertion per test is a habit worth building.

## 3. Asserting on Status, Headers, Body

Every interesting API test inspects some combination of three things:

1. **Status code** — did the route classify the request correctly? (200 vs 201 vs 404 vs 422 vs 409.)
2. **Headers** — content type, custom response headers (`Location` on a POST, `WWW-Authenticate` on a 401, rate-limit headers).
3. **Body** — the JSON payload, parsed and asserted against a dict (or list, or value).

To make the rest of the notebook concrete, here's a small CRUD app for `Asset`s — the same domain model we've used since chapter 1. We'll test it from this section on.

In [3]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

def build_app() -> FastAPI:
    # Factory function — used by tests so each test can get a fresh app
    # with a fresh in-memory store. No globals, no leaked state.
    store: dict[str, Asset] = {}
    app = FastAPI()

    @app.get("/assets", response_model=list[Asset])
    def list_assets():
        return list(store.values())

    @app.get("/assets/{ticker}", response_model=Asset)
    def get_asset(ticker: str):
        ticker = ticker.upper()
        if ticker not in store:
            raise HTTPException(status.HTTP_404_NOT_FOUND, f"Asset '{ticker}' not found")
        return store[ticker]

    @app.post("/assets", response_model=Asset, status_code=status.HTTP_201_CREATED)
    def create_asset(asset: Asset):
        if asset.ticker in store:
            raise HTTPException(status.HTTP_409_CONFLICT, f"Ticker '{asset.ticker}' exists")
        store[asset.ticker] = asset
        return asset

    @app.delete("/assets/{ticker}", status_code=status.HTTP_204_NO_CONTENT)
    def delete_asset(ticker: str):
        ticker = ticker.upper()
        if ticker not in store:
            raise HTTPException(status.HTTP_404_NOT_FOUND, f"Asset '{ticker}' not found")
        del store[ticker]

    return app

# Quick smoke against the factory.
smoke = TestClient(build_app())
print("empty list :", smoke.get("/assets").status_code, smoke.get("/assets").json())

empty list : 200 []


In [4]:
# Three small assertions exercising status, headers, body.
c = TestClient(build_app())

# --- status + body on the success path
created = c.post("/assets", json={"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0})
assert created.status_code == 201, created.text
assert created.json() == {"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0}

# --- headers on a JSON response: content-type is automatic
assert created.headers["content-type"].startswith("application/json")

# --- status + body on a sad path
missing = c.get("/assets/NVDA")
assert missing.status_code == 404
assert missing.json() == {"detail": "Asset 'NVDA' not found"}

# --- 204 has no body — assert that explicitly
deleted = c.delete("/assets/AAPL")
assert deleted.status_code == 204
assert deleted.content == b""

print("all three assertion layers OK")

all three assertion layers OK


A few habits these snippets quietly encode:

- **Always include `.text` (or `.json()`) in the failure message of a status assertion.** When CI fails with `assert 422 == 201`, you need the body to see *why*. `assert resp.status_code == 201, resp.text` is the one-line difference between a useful and a useless failure.
- **Don't share a client between unrelated tests.** Every cell above built a fresh app via `build_app()` so the dict store starts empty. We'll formalize this with fixtures next.
- **Compare whole dicts, not field-by-field.** `assert created.json() == {...}` catches extra fields you didn't expect (a regression where a route leaks an internal `created_by`). Field-by-field assertions miss that.

## 4. Fixtures: One Source of Setup

By section 3, we were calling `build_app()` and wrapping in `TestClient` in every test. That repetition is the cue to introduce **fixtures**. A fixture is a function decorated with `@pytest.fixture` that pytest calls *before* a test, passes by parameter name, and (optionally) tears down *after*.

Key levers:

- **`scope`** — `"function"` (default, rebuilt for every test, maximum isolation), `"module"` (built once per file), `"session"` (built once per test run). The tradeoff is speed vs cross-test contamination. Always start at `"function"`; only widen the scope when profiling shows it matters.
- **`conftest.py`** — fixtures defined here are visible to every test file in the same directory (and below). It's how you share a `client` fixture across a whole test suite without imports.
- **`yield` for cleanup** — a fixture that does `yield x` runs the post-yield code after the test returns, even if the test fails. This is the right place for `app.dependency_overrides.clear()` (notebook 7.3).

To prove the pytest layer end-to-end, we write a small project to a temp directory and shell out to `pytest`. That gives us the real CLI output a developer or CI sees.

In [5]:
import subprocess
import sys
from pathlib import Path
from tempfile import mkdtemp
from textwrap import dedent

def run_pytest(files: dict[str, str], *args: str) -> str:
    """Write each (filename -> source) into a fresh tempdir and run pytest there.

    Returns the combined stdout+stderr so the notebook can print it."""
    d = Path(mkdtemp(prefix="pytest_demo_"))
    for name, src in files.items():
        (d / name).write_text(dedent(src).lstrip(), encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", str(d), "-q", "--no-header", *args],
        capture_output=True, text=True,
    )
    return result.stdout + result.stderr

# The SUT (system under test) — the same Asset CRUD app, written as a real module
# so the test files can `from app import build_app`.
APP_PY = '''
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

def build_app() -> FastAPI:
    store: dict[str, Asset] = {}
    app = FastAPI()

    @app.get("/assets", response_model=list[Asset])
    def list_assets():
        return list(store.values())

    @app.get("/assets/{ticker}", response_model=Asset)
    def get_asset(ticker: str):
        ticker = ticker.upper()
        if ticker not in store:
            raise HTTPException(404, f"Asset '{ticker}' not found")
        return store[ticker]

    @app.post("/assets", response_model=Asset, status_code=201)
    def create_asset(asset: Asset):
        if asset.ticker in store:
            raise HTTPException(409, f"Ticker '{asset.ticker}' exists")
        store[asset.ticker] = asset
        return asset

    return app
'''

# conftest.py — fixtures shared across every test in this directory.
CONFTEST = '''
import pytest
from fastapi.testclient import TestClient
from app import build_app

@pytest.fixture
def app():
    # Fresh app per test => fresh in-memory store => total isolation.
    return build_app()

@pytest.fixture
def client(app):
    # Fixtures can depend on other fixtures by name.
    return TestClient(app)

@pytest.fixture
def aapl():
    return {"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0}
'''

# The actual test file — note how compact the bodies get.
TEST_BASIC = '''
def test_empty_list(client):
    response = client.get("/assets")
    assert response.status_code == 200
    assert response.json() == []

def test_create_then_get(client, aapl):
    created = client.post("/assets", json=aapl)
    assert created.status_code == 201, created.text
    fetched = client.get("/assets/AAPL")
    assert fetched.status_code == 200
    assert fetched.json() == aapl

def test_isolation_between_tests(client):
    # If fixtures leaked state, AAPL would still be here from the test above.
    # The empty list proves the function-scoped `app` fixture rebuilt the store.
    assert client.get("/assets").json() == []
'''

output = run_pytest({"app.py": APP_PY, "conftest.py": CONFTEST, "test_basic.py": TEST_BASIC})
print(output)

...                                                                      [100%]
============================== warnings summary ===============================
..\..\..\..\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1
  C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
3 passed, 1 warning in 0.58s



Read the output: three tests collected, three passed. The interesting one is `test_isolation_between_tests` — it asserts the store is empty *after* `test_create_then_get` inserted into it. That passes because `app` is function-scoped: pytest rebuilt the app (and its in-memory store) before the third test ran. Widen the scope to `"module"` and that test will fail, which is exactly the kind of bug fixture scope is designed to surface or hide.

## 5. Parametrize Happy and Sad Paths

Validation tests are *tabular* by nature: "for each of these bad inputs, expect status X and an error mentioning field Y." `@pytest.mark.parametrize` collapses the table into one function with one assertion block.

Two patterns to know:

- **Positional**: `@pytest.mark.parametrize("a, b, expected", [(1, 2, 3), (4, 5, 9)])`.
- **Named via `pytest.param(..., id="...")`**: gives each row a readable test ID in the output — invaluable when row 4 of 20 fails and you want to know *which* row 4 is.

The example below parametrizes both the happy path (one row of valid input → 201) and the sad path (several rows of invalid input → 422). The same function expresses both.

In [6]:
TEST_PARAM = '''
import pytest

@pytest.mark.parametrize(
    "payload",
    [
        pytest.param({"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0}, id="apple"),
        pytest.param({"ticker": "MSFT", "name": "Microsoft Corp.", "price": 420.0}, id="msft"),
        pytest.param({"ticker": "BRK.B", "name": "Berkshire Hathaway B", "price": 410.0}, id="brk-b-dot-allowed"),
    ],
)
def test_create_accepts_valid(client, payload):
    response = client.post("/assets", json=payload)
    assert response.status_code == 201, response.text
    assert response.json() == payload

@pytest.mark.parametrize(
    "bad_field, payload",
    [
        pytest.param("ticker",
            {"ticker": "lowercase", "name": "x", "price": 1.0},
            id="ticker-must-be-uppercase"),
        pytest.param("ticker",
            {"ticker": "TOOLONGTICKER", "name": "x", "price": 1.0},
            id="ticker-too-long"),
        pytest.param("name",
            {"ticker": "X", "name": "", "price": 1.0},
            id="name-empty"),
        pytest.param("price",
            {"ticker": "X", "name": "x", "price": -1.0},
            id="price-negative"),
    ],
)
def test_create_rejects_invalid(client, bad_field, payload):
    response = client.post("/assets", json=payload)
    assert response.status_code == 422, response.text
    # The 422 body contains a list of per-field errors; each has a `loc` tuple.
    # We assert that *the* bad field shows up at least once.
    offending = {tuple(err["loc"])[-1] for err in response.json()["detail"]}
    assert bad_field in offending, response.json()
'''

output = run_pytest({"app.py": APP_PY, "conftest.py": CONFTEST, "test_param.py": TEST_PARAM}, "-v")
print(output)

============================= test session starts =============================
collected 7 items

..\..\..\..\AppData\Local\Temp\pytest_demo_kmbtm3u4\test_param.py ...... [ 85%]
.                                                                        [100%]

============================== warnings summary ===============================
..\..\..\..\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1
  C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 7 passed, 1 warning in 0.61s =========================



The `-v` flag in this run is what gives you the readable per-case IDs: `test_create_rejects_invalid[ticker-too-long]` is far easier to grep than `test_create_rejects_invalid[1]`. The investment in `pytest.param(..., id="...")` pays back the first time a CI failure points at the wrong row.

One performance footnote: parametrized tests share fixtures *by scope*, not by row. With the function-scoped `client` fixture above, the app is rebuilt for *every* row — fine for a 4-row table, expensive at 4,000 rows. When you have a truly large table and the SUT is genuinely read-only for the test, scope your client fixture to `"module"` and you'll get one app for the whole table.

## 6. Markers, Selection, and the Slow Test Problem

Two things happen as a test suite matures:

1. Some tests get slow (a big parametrize, a long-running integration test, a benchmark).
2. You want to keep the inner loop snappy: edit code, run `pytest`, see green in two seconds — not two minutes.

Pytest's answer is **markers**. Tag the slow tests, then exclude them from the inner loop and run them only in CI (or before a commit). The mechanics:

- `@pytest.mark.slow` on a test function — purely a tag, no behavior change.
- `pytest -m "slow"` selects only marked tests; `pytest -m "not slow"` excludes them.
- You must **declare** custom markers in `pytest.ini` (or `pyproject.toml` under `[tool.pytest.ini_options]`) or pytest emits a `PytestUnknownMarkWarning`.

Related but different: `-k expr` selects by test *name*. `pytest -k "rejects and ticker"` runs only tests whose name matches both substrings. Useful for "I just edited the validator, only run validator tests."

In [7]:
TEST_MARKERS = '''
import pytest

def test_fast_one(client):
    assert client.get("/assets").status_code == 200

@pytest.mark.slow
def test_slow_one(client, aapl):
    # Imagine this is a 200-iteration concurrency stress test or
    # a real-DB integration test that takes seconds.
    # Two-letter tickers (AA, AB, ..., HR) stay inside the ^[A-Z.]{1,10}$ pattern.
    for i in range(200):
        ticker = chr(65 + (i // 26) % 26) + chr(65 + i % 26)
        client.post("/assets", json={**aapl, "ticker": ticker})
    assert len(client.get("/assets").json()) == 200
'''

PYTEST_INI = '''
[pytest]
markers =
    slow: marks tests as slow (deselect with -m "not slow")
'''

files = {
    "app.py": APP_PY, "conftest.py": CONFTEST,
    "test_markers.py": TEST_MARKERS, "pytest.ini": PYTEST_INI,
}

print("=== full run (both tests) ===")
print(run_pytest(files, "-v"))

print("=== inner loop: skip slow ===")
print(run_pytest(files, "-v", "-m", "not slow"))

print("=== CI only: slow tests ===")
print(run_pytest(files, "-v", "-m", "slow"))

=== full run (both tests) ===
============================= test session starts =============================
collected 2 items

..\..\..\..\AppData\Local\Temp\pytest_demo_3fvy6v2s\test_markers.py ..   [100%]

============================== warnings summary ===============================
..\..\..\..\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1
  C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 2 passed, 1 warning in 1.55s =========================

=== inner loop: skip slow ===
============================= test session starts =============================
collected 2 items / 1 deselected / 1 selected

..\..\..\..\AppData\Local\Temp\pytest_demo_as

Notice what the second run shows: `deselected: 1`. Pytest collected the slow test, matched it against the marker filter, and skipped it. The test count is honest about what ran and what was deferred — that honesty matters when a developer is checking "did I just run all the tests or only some?"

A trap to avoid: don't mark a test `slow` just because it's failing. If a flaky test gets quarantined behind a marker, no one runs it, no one fixes it, and a few months later you find it's been broken since you tagged it. Markers are for tiering by cost, not by reliability.

## Key Takeaways

- **`TestClient` is `httpx` pointed at your ASGI app in-process.** No server, no socket — the speed and simplicity of that is what makes a sub-second feedback loop achievable.
- **A response is `httpx.Response`.** Assert against `.status_code`, `.headers`, `.json()`. Pass `raise_server_exceptions=False` only when you specifically want to see the 500 a real client would see.
- **One logical assertion per test.** Status, headers, body each have a place, but keep each test focused on one observable outcome.
- **Fixtures centralize setup; scope is the speed/isolation lever.** Start at `function` scope, widen only when profiling justifies it. `conftest.py` shares fixtures across files.
- **`@pytest.mark.parametrize`** collapses tabular cases into one function. Give rows readable IDs with `pytest.param(..., id="...")`.
- **Markers tier tests by cost.** Declare them in `pytest.ini`, select with `-m`. Do not use markers to hide flaky tests.
- **Capstone tie-in**: the `client`/`app`/`sample_data` fixture trio in `conftest.py` becomes the foundation of `examples/portfolio_analytics_api/tests/`. Notebook 7.3 will swap in fake repositories at that fixture layer.

## Exercises

All exercises use the `run_pytest({...})` helper from section 4 — write the file dict and let pytest tell you whether it passes.

**1. Three CRUD tests for the Asset routes.** Write a `test_crud.py` that asserts: (a) `GET /assets/NVDA` returns 404 with a body whose `detail` contains the string `"NVDA"`; (b) `POST /assets` with the same ticker twice returns 201 then 409; (c) `POST /assets` with a payload missing `price` returns 422 and the response's `detail` list mentions `"price"` in some `loc`. Use the existing `client` and `aapl` fixtures from `conftest.py`.

**2. Promote a single test to `parametrize`.** Take exercise 1(a) and rewrite it as a parametrized test that covers three unknown tickers — `"NVDA"`, `"TSLA"`, `"XYZ"` — each verifying both status 404 and that the ticker appears in the body. Give each row a readable ID with `pytest.param(..., id="...")`.

**3. Add a `slow` test and verify selection.** Add `@pytest.mark.slow` to a test that creates 100 distinct assets in a loop, then lists them. Declare the marker in `pytest.ini`. Run pytest twice via the helper — once with `-m "not slow"` (expect the slow test to be deselected) and once with `-m "slow"` (expect only it to run). Confirm both runs end in `passed` rather than `error`.